# 🚀 S4D-RUPA-CLIP: Weakly-Supervised Video Anomaly Detection
### Standalone Residual-to-Text Counterfactual Alignment with S4D Temporal Modeling
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**Giảng viên hướng dẫn:** TS. Lê Viết Tuấn  
**Trường Đại học Mở TP. Hồ Chí Minh (HCMCOU)**  

---
### 🎯 Mục tiêu khoa học & Hiệu năng:
- **UCF-Crime:** Frame-level ROC-AUC >= 90.0%
- **XD-Violence:** Average Precision (AP) >= 87.0%
- **100% Standalone (Train from Scratch):** Triệt tiêu hoàn toàn DSANet, Dual GCN, Local Transformer và Warm-Start.
- **Kế thừa chuẩn hóa dữ liệu từ RUPA-DSA_final:** Đồng bộ và lập bản đồ đường dẫn tuyệt đối chính xác 100% cho cả 2 tập dữ liệu UCF-Crime và XD-Violence trước khi huấn luyện.

In [ ]:
# ========================= 1. THIẾT LẬP MÔI TRƯỜNG & TẢI MÃ NGUỒN =========================
REPO_URL = "https://github.com/Sharvuz/S4D-RUPA-CLIP.git"

import json, os, random, shutil, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "S4D-RUPA-CLIP"
ARTIFACTS = WORK / "s4d_rupa_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Clone hoặc Pull mã nguồn mới nhất
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

os.chdir(str(REPO))

# Cài đặt các thư viện bổ trợ
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ftfy", "regex", "scikit-learn", "pandas", "git+https://github.com/openai/CLIP.git"
], check=True)

import numpy as np
import pandas as pd
import torch

print("Repository:", REPO_URL)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print("GPU Count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Bật Accelerator = GPU trong Kaggle Settings!"


# =====================================================================
# PHẦN 1: HUẤN LUYỆN & KIỂM THỬ TRÊN UCF-CRIME
# Mục tiêu: ROC-AUC >= 90.0% | DeepMIL + 14 Multi-Prompt Ensembles
# =====================================================================

In [ ]:
# ========================= 2. CHUẨN HÓA DỮ LIỆU UCF-CRIME =========================
DATASET_UCF = "ucf"
print("🔍 Đang quét toàn bộ hệ thống Kaggle để tìm file .npy của UCF-Crime...")
t0 = time.time()

all_npy_paths = list(INPUT.rglob("*.npy")) + list(WORK.rglob("*.npy"))
if not all_npy_paths:
    raise FileNotFoundError("❌ CẢNH BÁO ĐỎ: Không tìm thấy file .npy nào! Hãy Add Input UCFClipFeatures vào Kaggle.")
else:
    print(f"✅ Radar đã quét thấy {len(all_npy_paths)} file .npy trên hệ thống trong {time.time()-t0:.2f}s!")

file_map = {path.name: path for path in all_npy_paths}

def rewrite_csv_ucf(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    
    missing = []
    is_first_line = True
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(",")
            feature_path = parts[0]
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            
            feature_basename = Path(feature_path).name
            possible_names = [
                feature_basename,
                feature_basename.replace("#", "+"),
                feature_basename.replace("+", "#"),
                feature_basename.replace("#", ""),
                feature_basename.replace("#", "_"),
            ]
            
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
            if not match_found:
                missing.append(feature_basename)
                
    if missing:
        raise FileNotFoundError(f"❌ Lỗi: Thiếu {len(missing)} file video trên Kaggle. Ví dụ: {missing[:5]}")
    print(f"  -> {output_name}: Đồng bộ thành công 100%! Không thiếu file nào.")
    return output

ucf_train_csv = rewrite_csv_ucf(f"list/{DATASET_UCF}_CLIP_rgb.csv", f"{DATASET_UCF}_train.csv")
ucf_test_csv = rewrite_csv_ucf(f"list/{DATASET_UCF}_CLIP_rgbtest.csv", f"{DATASET_UCF}_test.csv")

sample_path_ucf = Path(pd.read_csv(ucf_train_csv).iloc[0]["path"])
sample_ucf = np.load(sample_path_ucf)
assert sample_ucf.ndim == 2 and sample_ucf.shape[-1] == 512, f"Lỗi shape: {sample_ucf.shape}"
print(f"🎉 Feature preflight UCF hoàn hảo: {sample_path_ucf.name} | Shape: {sample_ucf.shape}")


In [ ]:
# ========================= 3. HUẤN LUYỆN S4D-RUPA-CLIP TRÊN UCF-CRIME =========================
MAX_EPOCHS_UCF = 15
BATCH_SIZE_UCF = 64
LR_UCF = 5e-5
NUM_WORKERS = 0

model_save_dir = str(ARTIFACTS / "models")
os.makedirs(model_save_dir, exist_ok=True)

cmd_ucf = [
    sys.executable, "-u", "src/ucf_train.py",
    "--train-list", str(ucf_train_csv),
    "--test-list", str(ucf_test_csv),
    "--dataset", "ucf",
    "--batch-size", str(BATCH_SIZE_UCF),
    "--lr", str(LR_UCF),
    "--max-epoch", str(MAX_EPOCHS_UCF),
    "--warmup-epochs", "2",
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--num-workers", str(NUM_WORKERS),
    "--use-text-prompt", "True",
    "--loss-rec-weight", "0.5",
    "--loss-sem-weight", "0.5",
    "--loss-bg-weight", "0.1",
    "--save-dir", model_save_dir
]

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN S4D-RUPA-CLIP TRÊN UCF-CRIME ({MAX_EPOCHS_UCF} Epochs)...\n")
subprocess.run(cmd_ucf, check=True)


In [ ]:
# ========================= 4. ĐÁNH GIÁ UCF-CRIME TEST SET =========================
best_model_ucf = os.path.join(model_save_dir, "s4d_rupa_clip_ucf_best.pth")

test_cmd_ucf = [
    sys.executable, "-u", "src/ucf_test.py",
    "--test-list", str(ucf_test_csv),
    "--model-path", best_model_ucf,
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--use-text-prompt", "True"
]

print("🔍 BẮT ĐẦU ĐÁNH GIÁ CHI TIẾT UCF-CRIME TEST SET...")
subprocess.run(test_cmd_ucf, check=True)


# =====================================================================
# PHẦN 2: HUẤN LUYỆN & KIỂM THỬ TRÊN XD-VIOLENCE
# Mục tiêu: AP >= 87.0% | Batch Size 96 | Tận dụng tối đa Dual-GPU T4
# =====================================================================

In [ ]:
# ========================= 5. CHUẨN HÓA DỮ LIỆU XD-VIOLENCE =========================
DATASET_XD = "xd"
print("🔍 Đang quét toàn bộ hệ thống Kaggle để tìm file .npy của XD-Violence...")
t0 = time.time()

all_npy_paths = list(INPUT.rglob("*.npy")) + list(WORK.rglob("*.npy"))
file_map = {path.name: path for path in all_npy_paths}
print(f"✅ Radar đã định vị {len(all_npy_paths)} file trên hệ thống trong {time.time()-t0:.2f}s!")

def rewrite_csv_xd(csv_rel, output_name):
    input_csv = REPO / csv_rel
    output = ARTIFACTS / output_name
    print(f"Đang đồng bộ danh sách {input_csv.name}...")
    
    missing = []
    is_first_line = True
    with input_csv.open("r", encoding="utf-8") as f, output.open("w", encoding="utf-8") as out:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(",")
            feature_path = parts[0]
            if is_first_line and ("path" in feature_path.lower() or "name" in feature_path.lower()):
                out.write(line + "\n")
                is_first_line = False
                continue
            is_first_line = False
            
            feature_basename = Path(feature_path).name
            possible_names = [
                feature_basename,
                feature_basename.replace("#", "+"),
                feature_basename.replace("+", "#"),
                feature_basename.replace("#", ""),
                feature_basename.replace("#", "_"),
            ]
            
            match_found = False
            for p_name in possible_names:
                if p_name in file_map:
                    parts[0] = str(file_map[p_name])
                    out.write(",".join(parts) + "\n")
                    match_found = True
                    break
            if not match_found:
                missing.append(feature_basename)
                
    if missing:
        raise FileNotFoundError(f"❌ Lỗi: Thiếu {len(missing)} file XD-Violence trên Kaggle. Ví dụ: {missing[:5]}")
    print(f"  -> {output_name}: Đồng bộ thành công 100% không thiếu file nào!")
    return output

xd_train_csv = rewrite_csv_xd(f"list/{DATASET_XD}_CLIP_rgb.csv", f"{DATASET_XD}_train.csv")
xd_test_csv = rewrite_csv_xd(f"list/{DATASET_XD}_CLIP_rgbtest.csv", f"{DATASET_XD}_test.csv")

sample_path_xd = Path(pd.read_csv(xd_train_csv).iloc[0]["path"])
sample_xd = np.load(sample_path_xd)
assert sample_xd.ndim == 2 and sample_xd.shape[-1] == 512, f"Lỗi shape: {sample_xd.shape}"
print(f"🎉 Preflight XD thành công: {sample_path_xd.name} | Shape: {sample_xd.shape}")


In [ ]:
# ========================= 6. HUẤN LUYỆN S4D-RUPA-CLIP TRÊN XD-VIOLENCE =========================
MAX_EPOCHS_XD = 20
BATCH_SIZE_XD = 96     # Batch size 96 tận dụng GPU tối đa cho tập XD-Violence lớn!
LR_XD = 5e-5

cmd_xd = [
    sys.executable, "-u", "src/xd_train.py",
    "--train-list", str(xd_train_csv),
    "--test-list", str(xd_test_csv),
    "--dataset", "xd",
    "--batch-size", str(BATCH_SIZE_XD),
    "--lr", str(LR_XD),
    "--max-epoch", str(MAX_EPOCHS_XD),
    "--warmup-epochs", "2",
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--num-workers", "0",
    "--use-text-prompt", "True",
    "--loss-rec-weight", "0.5",
    "--loss-sem-weight", "0.5",
    "--loss-bg-weight", "0.1",
    "--save-dir", str(ARTIFACTS / "models")
]

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN S4D-RUPA-CLIP TRÊN XD-VIOLENCE ({MAX_EPOCHS_XD} Epochs, Batch Size: {BATCH_SIZE_XD})...\n")
subprocess.run(cmd_xd, check=True)


In [ ]:
# ========================= 7. ĐÁNH GIÁ CHI TIẾT XD-VIOLENCE TEST SET =========================
best_model_xd = str(ARTIFACTS / "models" / "s4d_rupa_clip_xd_best.pth")

test_cmd_xd = [
    sys.executable, "-u", "src/xd_test.py",
    "--test-list", str(xd_test_csv),
    "--model-path", best_model_xd,
    "--num-layers", "2",
    "--d-state", "64",
    "--ssm-type", "s4d",
    "--num-prototypes", "16",
    "--use-text-prompt", "True"
]

print("🔍 BẮT ĐẦU ĐÁNH GIÁ CHI TIẾT XD-VIOLENCE TEST SET (Mục tiêu AP >= 87%)...")
subprocess.run(test_cmd_xd, check=True)
